# Fusion Model Fine-Tuning

Fine-tunes existing fusion models (LogReg + Meta-Classifier) on
`benchmark_ten_percent` using 4 submodels:
1. CNN Transfer (EfficientNet-B0)
2. ViT-Base
3. DeiT-Distilled
4. GradField CNN (CompactGradientNet v3)

---
## 1. Setup & Drive Mount

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import subprocess; subprocess.check_call(["pip", "install", "-q", "timm", "huggingface_hub", "joblib"])

In [ ]:
import os, sys, time, math, json, glob, random
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.transforms import functional as TF
import timm, joblib

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, confusion_matrix,
    roc_curve, average_precision_score
)
import numpy as np
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

if not torch.cuda.is_available():
    raise RuntimeError("❌ CUDA not available! Go to Runtime → Change runtime type → GPU")
device = "cuda"
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

---
## 2. Config

In [ ]:
ACTIVE_SUBMODELS = ["cnn-transfer", "vit-base", "deit-distilled", "gradfield-cnn"]
FUSION_THRESHOLD = 0.5
RANDOM_SEED = 42
HF_ORG = "DeepFakeDetector"

GDRIVE_DATA_DIR = "/content/drive/MyDrive/datasets"
GDRIVE_MODEL_DIR = Path("/content/drive/MyDrive/DeepfakeDetectionModels")
BENCHMARK_DIR = Path(GDRIVE_DATA_DIR) / "benchmark_ten_percent"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"}

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"Active submodels: {ACTIVE_SUBMODELS}")
print(f"Benchmark data: {BENCHMARK_DIR}")

---
## 3. Download All Models from HuggingFace

In [ ]:
from huggingface_hub import snapshot_download, login

HF_TOKEN = userdata.get('HF_TOKEN_WRITE')
if HF_TOKEN and HF_TOKEN.startswith("hf_"):
    login(token=HF_TOKEN)
else:
    print("⚠️ No HF token found. Private repos will fail.")

ALL_REPOS = [
    f"{HF_ORG}/cnn-transfer",
    f"{HF_ORG}/vit-base",
    f"{HF_ORG}/deit-distilled",
    f"{HF_ORG}/gradfield-cnn",
    f"{HF_ORG}/fusion-logreg",
    f"{HF_ORG}/fusion-meta-classifier",
]

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

for repo_id in ALL_REPOS:
    repo_name = repo_id.replace("/", "--")
    local_dir = GDRIVE_MODEL_DIR / repo_name
    print(f"\n📥 {repo_id} → {local_dir}")
    try:
        snapshot_download(repo_id=repo_id, local_dir=str(local_dir),
                          force_download=False, token=HF_TOKEN)
        print(f"   ✅ OK")
    except Exception as e:
        print(f"   ❌ {e}")

print("\n✅ All models downloaded.")

---
## 4. Model & Dataset Definitions

In [ ]:
class CompactGradientNet(nn.Module):
    def __init__(self, depth=4, base_filters=32, dropout=0.1, embedding_dim=128):
        super().__init__()
        self.N_GRAD_CHANNELS = 6
        sobel_x = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3)
        sobel_y = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32).view(1,1,3,3)
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)
        gaussian = torch.tensor([[1,4,6,4,1],[4,16,24,16,4],[6,24,36,24,6],
                                  [4,16,24,16,4],[1,4,6,4,1]], dtype=torch.float32) / 256.0
        self.register_buffer('gaussian', gaussian.view(1,1,5,5))
        self.input_norm = nn.BatchNorm2d(self.N_GRAD_CHANNELS)
        self.channel_mix = nn.Sequential(nn.Conv2d(self.N_GRAD_CHANNELS, self.N_GRAD_CHANNELS, 1), nn.ReLU())
        layers = []
        in_ch = self.N_GRAD_CHANNELS
        for i in range(depth):
            out_ch = base_filters * (2**i)
            layers.extend([nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(), nn.MaxPool2d(2)])
            if dropout > 0: layers.append(nn.Dropout2d(dropout))
            in_ch = out_ch
        self.cnn = nn.Sequential(*layers)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.embedding = nn.Linear(out_ch, embedding_dim)
        self.classifier = nn.Linear(embedding_dim, 1)

    def compute_gradient_field(self, luminance):
        G_x = F.conv2d(luminance, self.sobel_x, padding=1)
        G_y = F.conv2d(luminance, self.sobel_y, padding=1)
        mag = torch.sqrt(G_x**2 + G_y**2 + 1e-8)
        angle = torch.atan2(G_y, G_x)
        Gxx, Gxy, Gyy = G_x*G_x, G_x*G_y, G_y*G_y
        Sxx = F.conv2d(Gxx, self.gaussian, padding=2)
        Sxy = F.conv2d(Gxy, self.gaussian, padding=2)
        Syy = F.conv2d(Gyy, self.gaussian, padding=2)
        trace = Sxx + Syy
        det_term = torch.sqrt((Sxx-Syy)**2 + 4*Sxy**2 + 1e-8)
        l1, l2 = 0.5*(trace+det_term), 0.5*(trace-det_term)
        coh = ((l1-l2)/(l1+l2+1e-8))**2
        return torch.cat([G_x, G_y, torch.log1p(mag*10), torch.sin(angle), torch.cos(angle), coh], dim=1)

    def forward(self, luminance):
        x = self.compute_gradient_field(luminance)
        x = self.input_norm(x)
        x = self.channel_mix(x)
        x = self.cnn(x)
        x = self.global_pool(x).flatten(1)
        emb = self.embedding(x)
        logit = self.classifier(emb)
        return logit.squeeze(1), emb

In [ ]:
class LuminanceDataset(Dataset):
    def __init__(self, img_paths, labels, img_size=224):
        self.img_paths, self.labels = img_paths, labels
        self.resize = transforms.Resize((img_size, img_size))
        self.R, self.G, self.B = 0.2126, 0.7152, 0.0722
    def __len__(self): return len(self.img_paths)
    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert('RGB')
        img = self.resize(img)
        t = TF.to_tensor(img)
        lum = (self.R*t[0] + self.G*t[1] + self.B*t[2]).unsqueeze(0)
        return lum.float(), torch.tensor(self.labels[idx], dtype=torch.float32)


class StandardDataset(Dataset):
    def __init__(self, img_paths, labels, img_size=224):
        self.img_paths, self.labels = img_paths, labels
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)), transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
        ])
    def __len__(self): return len(self.img_paths)
    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert('RGB')
        return self.transform(img), torch.tensor(self.labels[idx], dtype=torch.float32)


def get_image_paths_and_labels(split_folder):
    """Collect images from benchmark split: {split}/{real,fake}/{source}/"""
    split_folder = Path(split_folder)
    if not split_folder.exists():
        print(f"⚠️ Not found: {split_folder}")
        return [], []
    paths, labels = [], []
    for label, cls in enumerate(["real", "fake"]):
        cls_dir = split_folder / cls
        if not cls_dir.exists(): continue
        for src in sorted(cls_dir.iterdir()):
            if src.is_dir():
                files = [f for f in src.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXTS]
                paths.extend([str(f) for f in files])
                labels.extend([label] * len(files))
                print(f"  {cls}/{src.name}: {len(files)}")
            elif src.is_file() and src.suffix.lower() in IMG_EXTS:
                paths.append(str(src)); labels.append(label)
    return paths, labels

---
## 5. Load Submodels

In [ ]:
def load_cnn_transfer(path):
    print(f"Loading cnn-transfer from {path}...")
    model = models.efficientnet_b0(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
    wp = os.path.join(path, "model.pth")
    if not os.path.exists(wp):
        files = glob.glob(f"{path}/*.pth")
        if files: wp = files[0]
    model.load_state_dict(torch.load(wp, map_location=device))
    return model.to(device).eval()

def load_vit_base(path):
    print(f"Loading vit-base from {path}...")
    model = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=2)
    model.head = nn.Sequential(nn.Linear(768,512), nn.ReLU(), nn.Dropout(0.1), nn.Linear(512,2))
    wp = os.path.join(path, "deepfake_vit_finetuned_wildfake.pth")
    if not os.path.exists(wp):
        files = glob.glob(f"{path}/*.pth")
        if files: wp = files[0]
    sd = torch.load(wp, map_location=device)
    if "model" in sd: sd = sd["model"]
    new_sd = {}
    for k, v in sd.items():
        if k.startswith("fc1."): k = k.replace("fc1.", "head.0.")
        elif k.startswith("fc2."): k = k.replace("fc2.", "head.3.")
        elif k.startswith("vit."): k = k.replace("vit.", "", 1)
        new_sd[k] = v
    model.load_state_dict(new_sd)
    return model.to(device).eval()

def load_deit_distilled(path):
    print(f"Loading deit-distilled from {path}...")
    model = timm.create_model('deit_base_distilled_patch16_224', pretrained=False)
    model.head = nn.Sequential(nn.LayerNorm(768), nn.Linear(768,512), nn.GELU(), nn.Dropout(0.2), nn.Linear(512,2))
    if hasattr(model, "head_dist"): model.head_dist = model.head
    wp = os.path.join(path, "deit_distilled_custom_head.pt")
    if not os.path.exists(wp):
        files = glob.glob(f"{path}/*.pt")
        if files: wp = files[0]
    model.load_state_dict(torch.load(wp, map_location=device), strict=False)
    return model.to(device).eval()

def load_gradfield_cnn(path):
    print(f"Loading gradfield-cnn from {path}...")
    model = CompactGradientNet()
    wp = os.path.join(path, "gradient_field_cnn_v2.pth")
    if not os.path.exists(wp):
        files = glob.glob(f"{path}/*.pth")
        if files: wp = files[0]
    sd = torch.load(wp, map_location=device, weights_only=False)
    if "model" in sd: sd = sd["model"]
    model.load_state_dict(sd)
    return model.to(device).eval()

# Load all submodels
model_paths = {
    "cnn-transfer":   str(GDRIVE_MODEL_DIR / f"{HF_ORG}--cnn-transfer"),
    "vit-base":       str(GDRIVE_MODEL_DIR / f"{HF_ORG}--vit-base"),
    "deit-distilled": str(GDRIVE_MODEL_DIR / f"{HF_ORG}--deit-distilled"),
    "gradfield-cnn":  str(GDRIVE_MODEL_DIR / f"{HF_ORG}--gradfield-cnn"),
}
loaders = {"cnn-transfer": load_cnn_transfer, "vit-base": load_vit_base,
           "deit-distilled": load_deit_distilled, "gradfield-cnn": load_gradfield_cnn}

loaded_models = {}
for name in ACTIVE_SUBMODELS:
    try:
        loaded_models[name] = loaders[name](model_paths[name])
        print(f"  ✅ {name}")
    except Exception as e:
        print(f"  ❌ {name}: {e}")

---
## 6. Load Benchmark Data

In [ ]:
print(f"📂 Loading from {BENCHMARK_DIR}\n")

print("--- Train ---")
train_paths, train_labels = get_image_paths_and_labels(BENCHMARK_DIR / "train")
print("\n--- Validation ---")
val_paths, val_labels = get_image_paths_and_labels(BENCHMARK_DIR / "validation")
print("\n--- Test ---")
test_paths, test_labels = get_image_paths_and_labels(BENCHMARK_DIR / "test")

ds_std_train = StandardDataset(train_paths, train_labels)
ds_lum_train = LuminanceDataset(train_paths, train_labels)
ds_std_val = StandardDataset(val_paths, val_labels)
ds_lum_val = LuminanceDataset(val_paths, val_labels)
ds_std_test = StandardDataset(test_paths, test_labels)
ds_lum_test = LuminanceDataset(test_paths, test_labels)

print(f"\n{'='*50}")
print(f"Train: {len(train_paths)} ({sum(train_labels)} fake)")
print(f"Val:   {len(val_paths)} ({sum(val_labels)} fake)")
print(f"Test:  {len(test_paths)} ({sum(test_labels)} fake)")

---
## 7. Collect Probabilities

In [ ]:
def get_probas(models_dict, ds_std, ds_lum, batch_size=32):
    dl_std = DataLoader(ds_std, batch_size=batch_size, shuffle=False, num_workers=2)
    dl_lum = DataLoader(ds_lum, batch_size=batch_size, shuffle=False, num_workers=2)
    probs_list, labels_list = [], []
    with torch.no_grad():
        for (img_std, y), (img_lum, _) in tqdm(zip(dl_std, dl_lum), total=len(dl_std)):
            img_std, img_lum = img_std.to(device), img_lum.to(device)
            batch_probs = []
            for name in ACTIVE_SUBMODELS:
                m = models_dict[name]
                if name == "gradfield-cnn":
                    logits, _ = m(img_lum)
                    p = torch.sigmoid(logits)
                elif name == "cnn-transfer":
                    p = torch.softmax(m(img_std), dim=1)[:, 1]
                else:
                    p = torch.softmax(m(img_std), dim=1)[:, 1]
                batch_probs.append(p.cpu().numpy())
            probs_list.append(np.stack(batch_probs, axis=1))
            labels_list.append(y.numpy())
    return np.concatenate(probs_list), np.concatenate(labels_list)

print("Collecting probabilities...")
X_train, y_train = get_probas(loaded_models, ds_std_train, ds_lum_train)
X_val, y_val = get_probas(loaded_models, ds_std_val, ds_lum_val)
X_test, y_test = get_probas(loaded_models, ds_std_test, ds_lum_test)

print(f"\nX_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

---
## 8. Fine-Tune LogReg

In [ ]:
FUSION_LOGREG_DIR = GDRIVE_MODEL_DIR / f"{HF_ORG}--fusion-logreg"
old_logreg_path = FUSION_LOGREG_DIR / "fusion_logreg.pkl"

print(f"Loading existing LogReg from {old_logreg_path}")
old_logreg = joblib.load(old_logreg_path)

old_n_features = old_logreg.coef_.shape[1]
new_n_features = len(ACTIVE_SUBMODELS)
print(f"Old features: {old_n_features}, New features: {new_n_features}")

# Create new LogReg with warm_start
fusion_logreg = LogisticRegression(random_state=RANDOM_SEED, warm_start=True, max_iter=1000)

if old_n_features == new_n_features:
    # Same dimensions — direct warm start
    print("✅ Same dimensions — direct weight transfer")
    fusion_logreg.coef_ = old_logreg.coef_.copy()
    fusion_logreg.intercept_ = old_logreg.intercept_.copy()
    fusion_logreg.classes_ = old_logreg.classes_.copy()
else:
    # Dimension mismatch (e.g. 3→4) — pad with zeros
    print(f"⚠️ Dimension mismatch ({old_n_features}→{new_n_features}) — padding coefficients")
    new_coef = np.zeros((1, new_n_features))
    # Copy old weights into first old_n_features columns
    n_copy = min(old_n_features, new_n_features)
    new_coef[0, :n_copy] = old_logreg.coef_[0, :n_copy]
    fusion_logreg.coef_ = new_coef
    fusion_logreg.intercept_ = old_logreg.intercept_.copy()
    fusion_logreg.classes_ = old_logreg.classes_.copy()

print(f"\nOld coefficients: {old_logreg.coef_[0]}")
print(f"Initial coefficients: {fusion_logreg.coef_[0]}")

# Fine-tune on new data
fusion_logreg.fit(X_train, y_train)

print(f"\n--- Fine-Tuned LogReg ---")
for name, coef in zip(ACTIVE_SUBMODELS, fusion_logreg.coef_[0]):
    print(f"  {name}: {coef:.4f}")
print(f"  Intercept: {fusion_logreg.intercept_[0]:.4f}")

# Evaluate
for split_name, X, y in [("Val", X_val, y_val), ("Test", X_test, y_test)]:
    probs = fusion_logreg.predict_proba(X)[:, 1]
    preds = (probs >= FUSION_THRESHOLD).astype(int)
    auroc = roc_auc_score(y, probs)
    acc = accuracy_score(y, preds)
    f1 = f1_score(y, preds)
    print(f"\n{split_name}: Acc={acc:.4f} F1={f1:.4f} AUROC={auroc:.4f}")

---
## 9. Fine-Tune MetaClassifier

In [ ]:
class MetaClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

FUSION_META_DIR = GDRIVE_MODEL_DIR / f"{HF_ORG}--fusion-meta-classifier"
old_meta_path = FUSION_META_DIR / "fusion_model.pt"

# Load old weights to inspect dimensions
old_state = torch.load(old_meta_path, map_location=device)
old_input_dim = old_state["net.0.weight"].shape[1]
print(f"Old MetaClassifier input_dim: {old_input_dim}")
print(f"New input_dim: {new_n_features}")

# Create new model
meta_model = MetaClassifier(new_n_features).to(device)

# Transfer weights
if old_input_dim == new_n_features:
    print("✅ Same dimensions — direct weight load")
    meta_model.load_state_dict(old_state)
else:
    print(f"⚠️ Dimension mismatch ({old_input_dim}→{new_n_features}) — transferring compatible weights")
    new_state = meta_model.state_dict()
    for key in old_state:
        if key == "net.0.weight":
            # First linear layer: (32, old) → (32, new), pad with zeros
            n_copy = min(old_input_dim, new_n_features)
            new_state[key][:, :n_copy] = old_state[key][:, :n_copy]
        elif key == "net.0.bias":
            new_state[key] = old_state[key]
        elif key in new_state and new_state[key].shape == old_state[key].shape:
            new_state[key] = old_state[key]
        else:
            print(f"  Skipping {key} (shape mismatch)")
    meta_model.load_state_dict(new_state)

print("Loaded weights, starting fine-tuning...\n")

# Fine-tune
FINETUNE_EPOCHS = 300
FINETUNE_LR = 1e-3

optimizer = torch.optim.Adam(meta_model.parameters(), lr=FINETUNE_LR)
criterion = nn.BCELoss()
X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)

meta_model.train()
for epoch in range(FINETUNE_EPOCHS):
    optimizer.zero_grad()
    loss = criterion(meta_model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:
        print(f"  Epoch {epoch}: Loss {loss.item():.4f}")

print(f"\n✅ MetaClassifier fine-tuned ({FINETUNE_EPOCHS} epochs)")

# Evaluate
meta_model.eval()
for split_name, X, y in [("Val", X_val, y_val), ("Test", X_test, y_test)]:
    with torch.no_grad():
        probs = meta_model(torch.tensor(X, dtype=torch.float32).to(device)).cpu().numpy().flatten()
    preds = (probs >= FUSION_THRESHOLD).astype(int)
    auroc = roc_auc_score(y, probs)
    acc = accuracy_score(y, preds)
    f1 = f1_score(y, preds)
    print(f"{split_name}: Acc={acc:.4f} F1={f1:.4f} AUROC={auroc:.4f}")

---
## 10. Comparison & Visualization

In [ ]:
# ROC overlay on test set
fig, ax = plt.subplots(figsize=(7, 6))

# LogReg
lr_probs = fusion_logreg.predict_proba(X_test)[:, 1]
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
auroc_lr = roc_auc_score(y_test, lr_probs)
ax.plot(fpr_lr, tpr_lr, lw=2, label=f"LogReg (AUROC={auroc_lr:.4f})")

# MetaClassifier
with torch.no_grad():
    mc_probs = meta_model(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy().flatten()
fpr_mc, tpr_mc, _ = roc_curve(y_test, mc_probs)
auroc_mc = roc_auc_score(y_test, mc_probs)
ax.plot(fpr_mc, tpr_mc, lw=2, label=f"MetaClassifier (AUROC={auroc_mc:.4f})")

ax.plot([0,1],[0,1],"k--",lw=1,alpha=0.5)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("Fusion Models — ROC on Test Set")
ax.legend(loc="lower right"); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"\n{'='*50}")
print(f"{'Model':<20} {'AUROC':>8} {'Acc':>8} {'F1':>8}")
print(f"{'-'*50}")
for name, probs in [("LogReg", lr_probs), ("MetaClassifier", mc_probs)]:
    preds = (probs >= FUSION_THRESHOLD).astype(int)
    print(f"{name:<20} {roc_auc_score(y_test, probs):>8.4f} "
          f"{accuracy_score(y_test, preds):>8.4f} {f1_score(y_test, preds):>8.4f}")
print(f"{'='*50}")

---
## 11. Export & Upload

In [ ]:
# Save LogReg
EXPORT_LR = Path("fusion-logreg-finetuned")
EXPORT_LR.mkdir(exist_ok=True)
joblib.dump(fusion_logreg, EXPORT_LR / "fusion_logreg.pkl")

config_lr = {
    "type": "probability_stacking_fusion", "name": "fusion-logreg",
    "version": "2.0.0-finetuned",
    "submodels": [f"{HF_ORG}/{n}" for n in ACTIVE_SUBMODELS],
    "submodel_order": ACTIVE_SUBMODELS,
    "num_submodels": len(ACTIVE_SUBMODELS),
    "threshold": FUSION_THRESHOLD,
    "labels": {"0": "real", "1": "fake"},
    "test_auroc": float(auroc_lr),
}
with open(EXPORT_LR / "config.json", "w") as f:
    json.dump(config_lr, f, indent=2)
with open(EXPORT_LR / "label_map.json", "w") as f:
    json.dump({"0": "real", "1": "fake"}, f)
print(f"✅ Exported LogReg to {EXPORT_LR}")

# Save MetaClassifier
EXPORT_MC = Path("fusion-meta-classifier-finetuned")
EXPORT_MC.mkdir(exist_ok=True)
torch.save(meta_model.state_dict(), EXPORT_MC / "fusion_model.pt")

config_mc = {
    "type": "probability_stacking_fusion", "name": "fusion-meta-classifier",
    "version": "2.0.0-finetuned",
    "submodels": [f"{HF_ORG}/{n}" for n in ACTIVE_SUBMODELS],
    "submodel_order": ACTIVE_SUBMODELS,
    "num_submodels": len(ACTIVE_SUBMODELS),
    "threshold": FUSION_THRESHOLD,
    "labels": {"0": "real", "1": "fake"},
    "test_auroc": float(auroc_mc),
}
with open(EXPORT_MC / "config.json", "w") as f:
    json.dump(config_mc, f, indent=2)
with open(EXPORT_MC / "label_map.json", "w") as f:
    json.dump({"0": "real", "1": "fake"}, f)
print(f"✅ Exported MetaClassifier to {EXPORT_MC}")

print(f"\n--- Upload Commands ---")
print(f"huggingface-cli upload {HF_ORG}/fusion-logreg ./{EXPORT_LR} . --create-repo")
print(f"huggingface-cli upload {HF_ORG}/fusion-meta-classifier ./{EXPORT_MC} . --create-repo")